# NANOGrav 15-year CW analysis with free-spectral pulsar noise

This notebook reproduces the QuickCW continuous-wave analysis of the NANOGrav 15-year dataset
(following [nanograv/15yr_cw_analysis](https://github.com/nanograv/15yr_cw_analysis)), but models the
**per-pulsar red noise with a free spectral model** (a free `log10_rho` parameter in each of 30
frequency bins, in every pulsar) while keeping the **power-law model for the GWB**.

The NG15 data files (per-pulsar enterprise `Pulsar` pickles, white-noise dictionary, and pulsar
distance priors) are included in this fork under `data/`, downloaded from the
[15yr_cw_analysis](https://github.com/nanograv/15yr_cw_analysis) repository.

Differences with respect to the published power-law analysis:

- `psr_noise_model='free_spectral'` is passed to `QuickCW` (the GWB stays `'powerlaw'`).
- This adds 67 pulsars x 30 bins = 2010 `{psr}_red_noise_log10_rho_{i}` parameters in place of the
  134 `gamma`/`log10_A` parameters.
- RN empirical-distribution proposals (`rn_emp_dist_file`) are specific to the power-law
  (`log10_A`, `gamma`) parametrization, so they are disabled here; red noise is sampled with
  diagonal Fisher-matrix jumps instead.

Setting everything up takes a few minutes and needs a good amount of RAM (10-20 GB).

In [ ]:
#imports
import numpy as np
import matplotlib.pyplot as plt

import glob
import bz2
import pickle

import QuickCW.QuickCW as QuickCW
from QuickCW.QuickMCMCUtils import ChainParams

In [ ]:
%%time
#loading in data from BZ2 compressed pickle files
psrs = []
for psrfile in sorted(glob.glob('data/jar/*.bz2.pkl')):
    with bz2.BZ2File(psrfile, 'rb') as f:
        psr = pickle.load(f)
        print(psr.name)
        psrs.append(psr)

print(len(psrs))

## Sampler settings

These mirror the settings of `running_quickcw.ipynb` in the 15yr_cw_analysis repo. As there, increase
`N` to 100 million - 1 billion for a production analysis (such runs typically take days).

In [ ]:
#number of iterations (increase to 100 million - 1 billion for actual analysis)
N = 5_000_000

n_int_block = 10_000      #number of iterations in a block (one shape update + projection updates)
save_every_n = 100_000    #number of iterations between saving intermediate results (integer multiple of n_int_block)
N_blocks = np.int64(N//n_int_block)
fisher_eig_downsample = 2000  #multiplier for how much less often to update fisher eigendirections

n_status_update = 100
n_block_status_update = np.int64(N_blocks//n_status_update)

assert N_blocks%n_status_update == 0
assert N%save_every_n == 0
assert N%n_int_block == 0

#Parallel tempering parameters
T_max = 3.
n_chain = 4

#white noise dictionary
noisefile = 'data/v1p1_all_dict.json'

#RN empirical distributions are defined for the powerlaw (log10_A, gamma) parameters,
#so they cannot be used with the free spectral pulsar noise model
rn_emp_dist_file = None

#parallax+DM based pulsar distance priors (psr objects in data/jar have zero distance and unit variance)
psr_dist_file = 'data/pulsar_distances_15yr.pkl'

#this is where results will be saved
savefile = 'quickCW_ng15_free_spectral_rn.h5'

chain_params = ChainParams(T_max, n_chain, n_block_status_update,
                           freq_bounds=np.array([np.nan, 3e-7]), #prior bounds on the GW frequency (np.nan -> 1/T_obs)
                           n_int_block=n_int_block,
                           save_every_n=save_every_n,
                           fisher_eig_downsample=fisher_eig_downsample,
                           rn_comps=30,   #number of frequency bins in the per-pulsar free spectral red noise model
                           gwb_comps=14,  #number of frequency components in the powerlaw GWB model
                           rn_emp_dist_file=rn_emp_dist_file,
                           savefile=savefile,
                           thin=100, #save every `thin`th sample to file
                           prior_draw_prob=0.2, de_prob=0.6, fisher_prob=0.3, #probability of different jump types
                           dist_jump_weight=0.2, rn_jump_weight=0.3, gwb_jump_weight=0.1,
                           common_jump_weight=0.2, all_jump_weight=0.2, #probability of updating different parameter groups
                           fix_rn=False, zero_rn=False, fix_gwb=False, zero_gwb=False)

## Model selection

The only change needed relative to the published analysis is `psr_noise_model='free_spectral'`
(the GWB stays a power law, which is also the default). The per-bin `log10_rho` prior is uniform
[-9, -4] and can be changed with `log10_rho_prior=np.array([low, high])`.

In [ ]:
%%time
pta, mcc = QuickCW.QuickCW(chain_params, psrs,
                           amplitude_prior='detection',     #'detection': uniform in log-amplitude, 'UL': uniform in amplitude
                           psr_distance_file=psr_dist_file, #parallax+DM pulsar distance priors
                           noise_json=noisefile,
                           psr_noise_model='free_spectral', #free spectral model for the per-pulsar red noise
                           gwb_noise_model='powerlaw')      #power-law model for the GWB

#the free spectral parameters of the first pulsar:
print([p for p in mcc.par_names if 'log10_rho' in p][:5], '...')
print('total number of parameters:', len(mcc.par_names))

## Running the MCMC

In [ ]:
%%time
mcc.advance_N_blocks(N_blocks)

## Looking at the results

The CW parameters can be analyzed exactly as in the `detection_analysis.ipynb` / `UL_analysis.ipynb`
notebooks of the 15yr_cw_analysis repo. Below we additionally show the free-spectral red noise
posterior for one pulsar.

In [ ]:
import h5py

with h5py.File(savefile, 'r') as f:
    par_names = [n.decode() for n in f['par_names'][:]]
    samples = f['samples_cold'][0, :, :]

#posterior of the CW frequency and amplitude
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(samples[:, par_names.index('0_log10_fgw')], bins=50, density=True)
axes[0].set_xlabel(r'$\log_{10} f_{\rm GW}$')
axes[1].hist(samples[:, par_names.index('0_log10_h')], bins=50, density=True)
axes[1].set_xlabel(r'$\log_{10} h$')
plt.tight_layout()
plt.show()

In [ ]:
#free spectral red noise posterior for one pulsar
psr_name = psrs[0].name
idx_rho = [i for i, n in enumerate(par_names) if n.startswith(psr_name + '_red_noise_log10_rho')]

Tspan = np.max([p.toas.max() for p in psrs]) - np.min([p.toas.min() for p in psrs])
freqs = np.arange(1, len(idx_rho) + 1) / Tspan

fig, ax = plt.subplots(figsize=(7, 4))
ax.violinplot([samples[:, i] for i in idx_rho], positions=np.log10(freqs), widths=0.04)
ax.set_xlabel(r'$\log_{10}(f/{\rm Hz})$')
ax.set_ylabel(r'$\log_{10}\rho$')
ax.set_title(f'{psr_name} free-spectral red noise posterior')
plt.show()